# 05a · Point sources with a redshift: input lists for visual inspection

Objects classified as point sources by SDSS (probPSF = 1) but with a measured redshift may be compact galaxies. This notebook prepares the SDSS image-list input for their visual inspection and converts the result (SourceFlag: 0 = invalid source, 1 = point source, 2 = extended source) into a CSV that `07_Photometry_Flag_update.ipynb` merges into the master catalog.

**Input**
- `A2199_mastercat_intermediate_file0.csv` – merged catalog from `03_merge_mastercat.py`

**Files**
- `05b_point_source_withz_SDSSimglist_input.txt` – image-list input (objid RA Dec)
- `05c_point_source_withz_SDSSimglist_template.txt` – result template; copied to `05c_point_source_withz_SDSSimglist_result.txt` and filled in by hand
- `05d_point_source_withz_SDSSimglist_result.csv` – the result as CSV

In [1]:
# ============================================================================
# Setup
# ============================================================================
import numpy as np
import pandas as pd

# Show every column when a DataFrame is displayed
pd.set_option('display.max_columns', None)

In [2]:
hecs_vac_data = pd.read_csv('../../DATA/AllHeCS_VAC_updated.csv')
Z_CLID = hecs_vac_data[hecs_vac_data['CLID'] == 'A2199']['Z'].values[0]

# Merged catalog (photometry + redshifts from all sources)
df0 = pd.read_csv('./A2199_mastercat_intermediate_file0.csv')

In [3]:
# Galactic-extinction-corrected model and Petrosian magnitudes (suffix "_0")
for band in ['u', 'g', 'r', 'i', 'z']:
    df0[f'p_modelmag_{band}_0'] = df0[f'p_modelmag_{band}'] - df0[f'p_extinction_{band}']
for band in ['u', 'g', 'r', 'i', 'z']:
    df0[f'p_petromag_{band}_0'] = df0[f'p_petromag_{band}'] - df0[f'p_extinction_{band}']

In [4]:
df0['grmod'] = df0['p_modelmag_g_0'] - df0['p_modelmag_r_0']

In [7]:
# Point sources (probPSF == 1) that have a redshift
point_withz = df0[(df0['z_tot_z'] != -9) & (df0['p_probpsf'] == 1)]

## Image-list input, result template, and conversion to CSV

In [ ]:
with open('05b_point_source_withz_SDSSimglist_input.txt', 'w') as f:
    temp = point_withz.sort_values(by='p_modelmag_r', ascending=True)
    for idx, row in temp.iterrows():
        f.write(f"{row['p_objid']} {row['p_ra']:.6f} {row['p_dec']:.6f}\n")

In [ ]:
# Template: every object starts as a point source (flag 1). Copy it to
# 05c_point_source_withz_SDSSimglist_result.txt and edit the flags by hand; the template
# is never read back, so re-running this notebook cannot overwrite the inspection result.
with open('05c_point_source_withz_SDSSimglist_template.txt', 'w') as f:
    f.write("objid RA DEC SourceFlag\n")
    f.write("SourceFlag 0 = Invalid Source 1 = Point Source 2 = Extended Source\n")
    temp = point_withz.sort_values(by='p_modelmag_r', ascending=True)

    for idx, row in temp.iterrows():
        f.write(f"{row['p_objid']},{row['p_ra']:.6f},{row['p_dec']:.6f}, 1\n")

In [ ]:
# Filled-in result -> CSV (the first two lines are the header and a note)
file_path = "./05c_point_source_withz_SDSSimglist_result.txt"

vis = pd.read_csv(
    file_path,
    sep=r"\s*,\s*",
    engine="python",
    skiprows=2,
    names=["objid", "RA", "DEC", "SourceFlag"],
)

vis["SourceFlag"] = vis["SourceFlag"].astype(int)
vis.to_csv("./05d_point_source_withz_SDSSimglist_result.csv", index=False)